# Listener Prior — Seq2Seq Keyphrase Generator (3 datasets, Option A) + fastT5 export

Trains **Flan‑T5‑Small** to generate **keyterms** and **keywords** for the next utterance (Option A).
Selects the best checkpoint by **VAL Recall@20 keyterms** and reports **both**:

- Recall@20 keyterms
- Recall@20 keywords

Saves to Drive + exports quantized ONNX via fastT5.


In [ ]:
# ---------------------------
# CONFIG
# ---------------------------
REPO_URL = "https://github.com/ebilal/fSTT.git"
PROJECT_DIR = "/content/listener-prior"

BASE_MODEL = "google/flan-t5-small"
HISTORY_TURNS = 6
MAX_KEYWORDS = 30
MAX_KEYTERMS = 30

EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-4

MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 64

TOPK = 20
SELECTION_FIELD = "val_recall@20_keyterms"

RUN_NAME = "seq2seq_flan_t5_small_optionA_3datasets_v2_sep_metrics"
DRIVE_ROOT = "/content/drive/MyDrive/listener_prior_runs"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
BEST_DIR = f"{RUN_DIR}/best_model"
FASTT5_DIR = f"{RUN_DIR}/best_fastT5_onnx"

RESTAURANT_CSV = "examples/multi_restaurant_phone_orders_2000.csv"


## Mount Drive + set HF_TOKEN from Colab Secrets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

for k in ['HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
    os.environ.pop(k, None)

if token:
    os.environ['HF_TOKEN'] = token
    os.environ['HUGGINGFACE_HUB_TOKEN'] = token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = token
    print('HF token set from Colab Secrets.')
else:
    print('No HF_TOKEN secret found.')


## Clone repo + install deps


In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

!python -m pip install -U pip
!grep -v '^torch' requirements.txt > /tmp/requirements_no_torch.txt
!python -m pip install -r /tmp/requirements_no_torch.txt --upgrade

!python -m pip install -U transformers accelerate datasets sentencepiece sacrebleu
!python -m pip install -U fastt5 onnx onnxruntime

import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())


## Load conversations + build Option A seq2seq pairs


In [ ]:
import os, random
import pandas as pd
from typing import List, Dict, Any
from src.prior import extract_priors

def load_public_conversations() -> List[Dict[str, Any]]:
    convos: List[Dict[str, Any]] = []
    try:
        from src import data as data_mod
        for fn_name in ["load_dual_dataset_conversations", "load_public_conversations", "get_dual_dataset"]:
            if hasattr(data_mod, fn_name):
                out = getattr(data_mod, fn_name)()
                if isinstance(out, dict):
                    for _, v in out.items():
                        if isinstance(v, list):
                            convos.extend(v)
                elif isinstance(out, list):
                    convos = out
                if convos:
                    print(f"Loaded public via src.data.{fn_name}: {len(convos)}")
                    return convos
    except Exception as e:
        print("Repo loader not available / failed:", repr(e))

    import datasets
    HF_DATASETS = [("daily_dialog", None), ("multi_woz_v22", None)]
    print("Using HF fallback datasets:", HF_DATASETS)

    for ds_name, cfg in HF_DATASETS:
        ds = datasets.load_dataset(ds_name) if cfg is None else datasets.load_dataset(ds_name, cfg)
        split = "train" if "train" in ds else list(ds.keys())[0]
        d = ds[split]
        if "dialog" in d.column_names:
            for i, dialog in enumerate(d["dialog"]):
                turns = [{"speaker": "unknown", "text": str(t)} for t in dialog]
                if len(turns) >= 2:
                    convos.append({"dialog_id": f"{ds_name}:{split}:{i}", "turns": turns})
        elif "turns" in d.column_names:
            for i, turns in enumerate(d["turns"]):
                norm = []
                if isinstance(turns, list) and turns and isinstance(turns[0], dict):
                    for t in turns:
                        txt = t.get("utterance") or t.get("text") or t.get("content") or ""
                        spk = t.get("speaker") or t.get("role") or "unknown"
                        if txt:
                            norm.append({"speaker": str(spk), "text": str(txt)})
                if len(norm) >= 2:
                    convos.append({"dialog_id": f"{ds_name}:{split}:{i}", "turns": norm})
    print("Loaded HF public conversations:", len(convos))
    return convos

def load_restaurant_csv(path: str) -> List[Dict[str, Any]]:
    assert os.path.exists(path), f"Missing CSV at {path}."
    df = pd.read_csv(path)
    required = {"dialog_id", "utterance_id", "speaker", "text"}
    assert required.issubset(set(df.columns)), f"CSV must include {required}, got {set(df.columns)}"
    convos: List[Dict[str, Any]] = []
    for did, g in df.sort_values(["dialog_id", "utterance_id"]).groupby("dialog_id"):
        turns = [{"speaker": str(r["speaker"]), "text": str(r["text"])} for _, r in g.iterrows()]
        if len(turns) >= 2:
            convos.append({"dialog_id": f"restaurant:{did}", "turns": turns})
    print("Loaded restaurant conversations:", len(convos))
    return convos

all_convos = load_public_conversations() + load_restaurant_csv(RESTAURANT_CSV)

def convo_to_pairs(convo: Dict[str, Any], history_turns: int) -> List[Dict[str, str]]:
    turns = convo["turns"]
    out = []
    for t in range(1, len(turns)):
        hist = turns[max(0, t-history_turns):t]
        target = turns[t]
        inp = "predict keyterms for next utterance\n" + "\n".join([f'{h["speaker"]}: {h["text"]}' for h in hist]).strip()
        pri = extract_priors(target["text"], max_keywords=MAX_KEYWORDS, max_keyterms=MAX_KEYTERMS)
        tgt = "keyterms: " + "; ".join(pri.get("keyterms", [])) + "\n"
        tgt += "keywords: " + "; ".join(pri.get("keywords", []))
        out.append({"dialog_id": convo["dialog_id"], "input_text": inp, "target_text": tgt})
    return out

pairs=[]
for c in all_convos:
    pairs.extend(convo_to_pairs(c, HISTORY_TURNS))
print("TOTAL pairs:", len(pairs))


## Split by dialog id


In [ ]:
from collections import defaultdict
import random
rng = random.Random(7)

by_dialog = defaultdict(list)
for p in pairs:
    by_dialog[p["dialog_id"]].append(p)

dialog_ids = list(by_dialog.keys())
rng.shuffle(dialog_ids)

n=len(dialog_ids)
n_train=int(0.8*n)
n_val=int(0.1*n)

train_ids=set(dialog_ids[:n_train])
val_ids=set(dialog_ids[n_train:n_train+n_val])
test_ids=set(dialog_ids[n_train+n_val:])

train_pairs=[p for did in train_ids for p in by_dialog[did]]
val_pairs=[p for did in val_ids for p in by_dialog[did]]
test_pairs=[p for did in test_ids for p in by_dialog[did]]

print("pairs train/val/test:", len(train_pairs), len(val_pairs), len(test_pairs))


## Recall@20 split: keyterms vs keywords


In [ ]:
def parse_keyterms_keywords(text: str):
    keyterms=[]
    keywords=[]
    for line in text.splitlines():
        line=line.strip()
        if line.lower().startswith("keyterms:"):
            rhs=line.split(":",1)[1]
            keyterms=[t.strip() for t in rhs.split(";") if t.strip()]
        elif line.lower().startswith("keywords:"):
            rhs=line.split(":",1)[1]
            keywords=[t.strip() for t in rhs.split(";") if t.strip()]
    def dedup(xs):
        seen=set(); out=[]
        for x in xs:
            if x not in seen:
                seen.add(x); out.append(x)
        return out
    return dedup(keyterms), dedup(keywords)

def recall_at_k(gt, pred, k=20):
    gt_set=set(gt)
    if not gt_set:
        return 0.0
    return len(set(pred[:k]) & gt_set) / len(gt_set)

def split_recalls(gt_text: str, pred_text: str, k=20):
    gt_t, gt_w = parse_keyterms_keywords(gt_text)
    pr_t, pr_w = parse_keyterms_keywords(pred_text)
    return {
        "recall@20_keyterms": recall_at_k(gt_t, pr_t, k),
        "recall@20_keywords": recall_at_k(gt_w, pr_w, k),
    }


## Train Flan‑T5 Small + select best by VAL Recall@20 keyterms


In [ ]:
import os, json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from torch.optim import AdamW
from tqdm.auto import tqdm

os.makedirs(RUN_DIR, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

class PairDS(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    inputs=[b["input_text"] for b in batch]
    targets=[b["target_text"] for b in batch]
    enc=tokenizer(inputs, max_length=MAX_INPUT_TOKENS, truncation=True, padding=True, return_tensors="pt")
    lab=tokenizer(targets, max_length=128, truncation=True, padding=True, return_tensors="pt")["input_ids"]
    lab[lab==tokenizer.pad_token_id] = -100
    enc["labels"]=lab
    return enc

train_dl=DataLoader(PairDS(train_pairs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
opt=AdamW(model.parameters(), lr=LR)

@torch.no_grad()
def eval_recalls(m, rows, max_batches=80):
    m.eval()
    sum_t=0.0; sum_w=0.0; n=0
    dl = DataLoader(PairDS(rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    for bi, batch in enumerate(dl):
        if bi >= max_batches:
            break
        inputs={k:v.to(device) for k,v in batch.items() if k!="labels"}
        gen=m.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1)
        pred_txt = tokenizer.batch_decode(gen, skip_special_tokens=True)
        start=bi*BATCH_SIZE
        gts=[r["target_text"] for r in rows[start:start+len(pred_txt)]]
        for g,p in zip(gts, pred_txt):
            m2 = split_recalls(g,p,k=TOPK)
            sum_t += m2["recall@20_keyterms"]
            sum_w += m2["recall@20_keywords"]
            n += 1
    return {"recall@20_keyterms": float(sum_t/max(1,n)), "recall@20_keywords": float(sum_w/max(1,n))}

best={SELECTION_FIELD:-1.0,"epoch":None}
history=[]

for epoch in range(1, EPOCHS+1):
    model.train()
    pbar=tqdm(train_dl, desc=f"epoch {epoch}/{EPOCHS}")
    total=0.0
    for batch in pbar:
        batch={k:v.to(device) for k,v in batch.items()}
        out=model(**batch)
        loss=out.loss
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        total += loss.item()
        pbar.set_postfix(loss=total/(pbar.n+1))

    val_m = eval_recalls(model, val_pairs, max_batches=80)
    print(f"VAL Recall@20 keyterms: {val_m['recall@20_keyterms']:.6f} | keywords: {val_m['recall@20_keywords']:.6f}")
    history.append({"epoch":epoch, "val_recall@20_keyterms": val_m["recall@20_keyterms"], "val_recall@20_keywords": val_m["recall@20_keywords"]})

    if val_m["recall@20_keyterms"] > best[SELECTION_FIELD]:
        best={SELECTION_FIELD: val_m["recall@20_keyterms"], "epoch": epoch, **val_m}
        os.makedirs(BEST_DIR, exist_ok=True)
        model.save_pretrained(BEST_DIR)
        tokenizer.save_pretrained(BEST_DIR)
        with open(os.path.join(RUN_DIR,"best_metrics.json"),"w") as f:
            json.dump({"selection_metric": SELECTION_FIELD, **best, "history": history}, f, indent=2)
        print("✓ Saved BEST model:", BEST_DIR)

print("BEST:", best)


## Test evaluation + save performance.json


In [ ]:
import os, json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

best_tok = AutoTokenizer.from_pretrained(BEST_DIR)
best_model = AutoModelForSeq2SeqLM.from_pretrained(BEST_DIR).to(device)

@torch.no_grad()
def eval_test(m, rows, max_batches=200):
    m.eval()
    sum_t=0.0; sum_w=0.0; n=0
    dl = DataLoader(PairDS(rows), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    for bi, batch in enumerate(dl):
        if bi >= max_batches:
            break
        inputs={k:v.to(device) for k,v in batch.items() if k!="labels"}
        gen=m.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1)
        pred_txt = best_tok.batch_decode(gen, skip_special_tokens=True)
        start=bi*BATCH_SIZE
        gts=[r["target_text"] for r in rows[start:start+len(pred_txt)]]
        for g,p in zip(gts, pred_txt):
            m2=split_recalls(g,p,k=TOPK)
            sum_t += m2["recall@20_keyterms"]
            sum_w += m2["recall@20_keywords"]
            n += 1
    return {"recall@20_keyterms": float(sum_t/max(1,n)), "recall@20_keywords": float(sum_w/max(1,n))}

test_m = eval_test(best_model, test_pairs, max_batches=200)
print(f"TEST Recall@20 keyterms: {test_m['recall@20_keyterms']:.6f} | keywords: {test_m['recall@20_keywords']:.6f}")

perf={
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "history_turns": HISTORY_TURNS,
    "selection_metric": SELECTION_FIELD,
    "best_epoch": best.get("epoch"),
    "best_val_recall@20_keyterms": best.get("recall@20_keyterms"),
    "best_val_recall@20_keywords": best.get("recall@20_keywords"),
    "test_recall@20_keyterms": test_m["recall@20_keyterms"],
    "test_recall@20_keywords": test_m["recall@20_keywords"],
}
with open(os.path.join(RUN_DIR,"performance.json"),"w") as f:
    json.dump(perf,f,indent=2)
print("Saved:", os.path.join(RUN_DIR,"performance.json"))


## Export BEST to fastT5 (quantized ONNX)


In [ ]:
import os
from fastT5 import export_and_get_onnx_model

os.makedirs(FASTT5_DIR, exist_ok=True)
onnx_model = export_and_get_onnx_model(BEST_DIR, onnx_dir=FASTT5_DIR, quantize=True)
print("Saved fastT5 artifacts:", FASTT5_DIR)


## Example predictions (2 test examples)


In [ ]:
def generate_pred(history_text: str) -> str:
    enc = best_tok([history_text], truncation=True, padding=True, max_length=MAX_INPUT_TOKENS, return_tensors="pt").to(device)
    gen = best_model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, num_beams=1)
    return best_tok.batch_decode(gen, skip_special_tokens=True)[0]

for p in test_pairs[:2]:
    pred = generate_pred(p["input_text"])
    gt_t, gt_w = parse_keyterms_keywords(p["target_text"])
    pr_t, pr_w = parse_keyterms_keywords(pred)

    ov_t = sorted(set(pr_t[:20]) & set(gt_t))
    ov_w = sorted(set(pr_w[:20]) & set(gt_w))

    print("="*90)
    print("HISTORY:\n", p["input_text"][:800])
    print("\nPRED text:\n", pred)

    print("\nGT keyterms:", gt_t[:30])
    print("PRED keyterms@20:", pr_t[:20])
    print("Overlap keyterms:", ov_t, f"(recall={len(ov_t)}/{len(set(gt_t)) if gt_t else 0})")

    print("\nGT keywords:", gt_w[:30])
    print("PRED keywords@20:", pr_w[:20])
    print("Overlap keywords:", ov_w, f"(recall={len(ov_w)}/{len(set(gt_w)) if gt_w else 0})")


## Production usage (seq2seq)

The generated text contains two lines (`keyterms:` and `keywords:`). In production, parse them separately with `parse_keyterms_keywords(...)`
and take the first 20 of each.

For real-time latency:
- `num_beams=1`
- keep `max_new_tokens` small
- cap input length
- prefer fastT5 ONNX for CPU.
